# BIAS-PROFS データセットのグラフ表示画像およびラベルCSVの作成

## 1. 必要ライブラリのインポート

In [1]:
from Bio import SeqIO
from PIL import Image
from tqdm import tqdm
import csv
import numpy as np
import os
import shutil

## 2. BIAS-PROFS データセットの読み込み

In [2]:
biasprofs_path = "../data/gds_dataset"  # BIAS-PROFS データセットの保管場所
biasprofs_excluded_path = "../data/gds_dataset/excluded"  # 除外されたデータセットの保管場所
os.makedirs(biasprofs_excluded_path, exist_ok=True)  # 除外されたデータセットの保管場所の作成

if os.path.exists(os.path.join(biasprofs_path, "readme.txt")):  # readme.txt を削除する
    os.remove(os.path.join(biasprofs_path, "readme.txt"))

paths = [os.path.join(biasprofs_path, f) for f in os.listdir(biasprofs_path) if f.endswith(".txt")]  # パスのリスト

### 2.1 10 件以下のファイルを除外
先行研究に倣って 10 例以下のファミリを除外します

除外された `.txt` ファイルは `📂 ../data/gds_dataset/excluded` に移動します

In [3]:
for path in paths:
    with open(path, "r", encoding="utf-8") as handle:
        l = len(list(SeqIO.parse(handle, "fasta")))
        print(f"{path}: {l}")

    if l <= 10:  # 10 例以下は excluded フォルダに移動
        shutil.move(path, os.path.join(biasprofs_excluded_path, os.path.basename(path)))

../data/gds_dataset\ClassA_Adrenergic_Adrenergic.txt: 95
../data/gds_dataset\ClassA_Amine_Adrenoreceptor.txt: 174
../data/gds_dataset\ClassA_Amine_Dopamine.txt: 228
../data/gds_dataset\ClassA_Amine_Histamine.txt: 53
../data/gds_dataset\ClassA_Amine_MuscAcetyl.txt: 159
../data/gds_dataset\ClassA_Amine_Muscarinicacetylcholine.txt: 186
../data/gds_dataset\ClassA_Amine_Octopamine.txt: 37
../data/gds_dataset\ClassA_Amine_Serotonin.txt: 503
../data/gds_dataset\ClassA_Amine_Traceamine.txt: 149
../data/gds_dataset\ClassA_Anaphylatoxin_Anaphylatoxin.txt: 30
../data/gds_dataset\ClassA_Cannabinoid_Cannabinoid.txt: 151
../data/gds_dataset\ClassA_Fmetleuphe_Fmetleuphe.txt: 7
../data/gds_dataset\ClassA_GRHR_AKH.txt: 8
../data/gds_dataset\ClassA_GRHR_Cora.txt: 6
../data/gds_dataset\ClassA_GRHR_GRHR.txt: 91
../data/gds_dataset\ClassA_Hormone_FollicleStim.txt: 66
../data/gds_dataset\ClassA_Hormone_Gonadotrophin.txt: 93
../data/gds_dataset\ClassA_Hormone_Lutropin.txt: 9
../data/gds_dataset\ClassA_Hormon

## 2.2 クラス分布の確認

In [4]:
aa_list = list("ACDEFGHIKLMNPQRSTVWY")
exclude_aa = list("BJOUXZ")

def check_valid_seq(seq):
    """
    有効なタンパク質かどうかを返す関数
    :param seq: タンパク質配列
    :return: 有効なタンパク質ならば True そうでないならば False
    """
    if len(seq) >= 1000:  # 配列長が 1000 以上のものを除外
        return False

    for aa in exclude_aa:  # BJOUXZ が含まれていたら除外
        if aa in seq:
            return False

    return True

In [5]:
biasprofs_path = "../data/gds_dataset"  # BIAS-PROFS データセットの保管場所
paths = [os.path.join(biasprofs_path, f) for f in os.listdir(biasprofs_path) if f.endswith(".txt")]  # パスのリスト

In [6]:
class_count = {cls: 0 for cls in list("ABCDE")}  # クラスに含まれてる配列数のカウント
aa_count = {aa: 0 for aa in aa_list}  # アミノ酸ごとのカウント
total_seq = 0  # 全ての配列数のカウント
total_aa = 0  # 全てのアミノ酸の数のカウント

for path in paths:
    cls_name = os.path.basename(path).split("Class")[1][0]  # クラス名の取得

    with open(path, "r", encoding="utf-8") as handle:
        for record in SeqIO.parse(handle, "fasta"):
            if check_valid_seq(record.seq):  # 有効な配列かどうか
                seq = record.seq

                class_count[cls_name] += 1
                total_seq += 1
                total_aa += len(seq)

                for aa in seq:
                    aa_count[aa] += 1

print(f"class_count = {class_count}")
print(f"aa_count = {aa_count}")
print(f"total_seq = {total_seq}")
print(f"total_aa = {total_aa}")

class_count = {'A': 5302, 'B': 454, 'C': 1929, 'D': 13, 'E': 18}
aa_count = {'A': 240437, 'C': 105093, 'D': 113558, 'E': 124238, 'F': 202338, 'G': 172916, 'H': 77062, 'I': 239917, 'K': 143943, 'L': 400594, 'M': 92656, 'N': 145011, 'P': 159723, 'Q': 104298, 'R': 160883, 'S': 279569, 'T': 204290, 'V': 259207, 'W': 65832, 'Y': 120500}
total_seq = 7716
total_aa = 3412065


## 3. グラフ表示画像の作成

In [7]:
SIZE = 224  # 画像サイズ
PADDING = 3  # パディング
LINEWIDTH = 2  # 線の太さ
IMAGE_SAVE_DIR = "../images/bias-profs"  # 画像の保存場所

os.makedirs(IMAGE_SAVE_DIR, exist_ok=True)  # ディレクトリの作成

# アミノ酸のベクトル（詳細は論文参照）
amino_vec = {
    "A": {"x": -0.99019942, "y": 2.29554027},
    "R": {"x":  7.38605815, "y": 1.30236133},
    "N": {"x":  4.01061596, "y": 2.98579296},
    "D": {"x":  2.04788011, "y": 1.43394109},
    "C": {"x": -2.18212092, "y": 2.05872491},
    "Q": {"x":  4.99366066, "y": 3.32616193},
    "E": {"x":  3.73512536, "y": 3.32397933},
    "G": {"x":  0.04937042, "y": 0.49755659},
    "H": {"x":  4.48215043, "y": 3.98877519},
    "I": {"x": -5.41644264, "y": 0.95506498},
    "L": {"x": -5.11141138, "y": 2.03063381},
    "K": {"x":  6.26454053, "y": 3.12338469},
    "M": {"x": -3.58295155, "y": 4.81273916},
    "F": {"x": -5.09870182, "y": 4.03153069},
    "P": {"x":  2.46839549, "y": 4.91497952},
    "S": {"x":  0.17443449, "y": 2.99492447},
    "T": {"x":  0.58046457, "y": 4.96619179},
    "W": {"x":  0.32567719, "y": 6.99241978},
    "Y": {"x":  2.00762263, "y": 6.70592659},
    "V": {"x": -4.82962913, "y": 1.29409523},
}

def map_range(x, a, b, c, d):
    """
    変数 $x$ を 範囲 [$a$, $b$] から [$c$, $d$] にリマップします
    :param x: 変換する対象
    :param a: 変換前の下端
    :param b: 変換前の上端
    :param c: 変換後の下端
    :param d: 変換後の上端
    :return: リマップした値
    """
    return (x - a) / (b - a) * (d - c) + c


def draw_point(canvas, x: int, y: int, width: int, intensity: int):
    """
    点を描画します
    :param canvas: numpy の2次元配列
    :param x: 座標 $x$
    :param y: 座標 $y$
    :param width: 点の幅
    :param intensity: 点の白色の強度
    :return:
    """

    size = canvas.shape[0]

    for i in range(-width, width + 1):
        yi = y + i

        if 0 <= yi < size:
            for j in range(-width, width + 1):
                xi = x + j

                if 0 <= xi < size:
                    r = np.sqrt(i ** 2 + j ** 2)  # 中心からの距離

                    if r <= width:  # 半径内部かどうかを判定
                        val = int(intensity * (1 - r / width))  # 直線的に色を変化
                        # val = int((intensity / (r_0 + 1)) - 1)  # 中心から反比例するように色を変化

                        if val > canvas[yi, xi]:  # 画素よりも大きい値であれば上書き
                            canvas[yi, xi] = val


def drawline_with_bresenham_algorithm(canvas, x0: int, y0: int, x1: int, y1: int, width: int, intensity: int):
    """
    ブレゼンハムのアルゴリズムで2点間の直線を描画します
    :param canvas: numpy の2次元配列
    :param x0: 点 $x_0$
    :param y0: 点 $y_0$
    :param x1: 点 $x_1$
    :param y1: 点 $y_1$
    :param width 直線の幅
    :return: None
    """

    # ここで float を int に変換
    x0 = round(x0)
    y0 = round(y0)
    x1 = round(x1)
    y1 = round(y1)

    dx = abs(x1 - x0)
    dy = abs(y1 - y0)
    sx = 1 if x0 < x1 else -1
    sy = 1 if y0 < y1 else -1
    err = dx - dy

    while True:
        draw_point(canvas, x0, y0, width, intensity)

        if x0 == x1 and y0 == y1:
            break
        e2 = 2 * err

        if e2 > -dy:
            err -= dy
            x0 += sx

        if e2 < dx:
            err += dx
            y0 += sy

    draw_point(canvas, x1, y1, width, intensity)


def generate_graph(seq, accession_number, size, padding, width):
    """
    アミノ酸配列からグラフ表示画像を作成します
    :param seq: アミノ酸配列
    :param accession_number: アクセッション番号
    :param size: 画像サイズ
    :param padding: 画像のパディング
    :param width: 線幅
    :return:
    """

    x = y = 0
    x_min = x_max = 0
    y_min = y_max = 0

    points = [{"x": 0, "y": 0}]

    for c in seq:
        x += amino_vec[c]["x"]
        y += amino_vec[c]["y"]  # 人間が見るXY平面にする

        points.append({"x": x, "y": y})

        x_min, x_max = min(x, x_min), max(x, x_max)
        y_min, y_max = min(y, y_min), max(y, y_max)

    mapped_points = [
        {
            "x": map_range(point["x"], x_min, x_max, padding, (size - padding)),
            "y": size - map_range(point["y"], y_min, y_max, padding, (size - padding))  # xy 平面にするために反転
        }
        for point in points
    ]

    canvas = np.zeros((size, size), dtype=np.uint8)
    n_segments = len(mapped_points) - 1

    for i in range(n_segments):
        start, end = mapped_points[i], mapped_points[i + 1]

        progress = i / n_segments
        intensity = int(255 * progress)
        # draw.line([(start["x"], start["y"]), (end["x"], end["y"])], fill=255, width=3)
        drawline_with_bresenham_algorithm(canvas, start["x"], start["y"], end["x"], end["y"], width, intensity)

    img = Image.fromarray(canvas, mode="L")  # numpy 配列から画像を生成
    img.save(os.path.join(IMAGE_SAVE_DIR, f"{accession_number}.png"))

In [8]:
n = 1

for path in tqdm(paths):
    cls_name = os.path.basename(path).split("Class")[1][0]  # クラス名の取得

    with open(path, "r", encoding="utf-8") as handle:
        for record in SeqIO.parse(handle, "fasta"):
            if check_valid_seq(record.seq):  # 有効な配列かどうか
                seq = record.seq
                generate_graph(seq, n, SIZE, PADDING, LINEWIDTH)  # グラフ表示画像の作成

                n += 1

100%|██████████| 87/87 [06:29<00:00,  4.48s/it]


## 4. ラベルCSVの作成

以下のような形式になります

|  ID  | Class |
|:----:|:-----:|
|  1   |   A   |
|  2   |   A   |
| ...  |  ...  |
| 7715 |   E   |
| 7716 |   E   |

In [9]:
labels_path = "../labels"  # ラベル保管場所
os.makedirs(labels_path, exist_ok=True)  # ディレクトリ作成

csv_path = os.path.join(labels_path, "bias-profs-labels.csv")  # CSV ファイルのパス

In [10]:
idx = 1

with open(csv_path, "w", encoding="utf-8", newline="") as csv_file:
    writer = csv.writer(csv_file)
    writer.writerow(["ID", "Class"])  # ヘッダー

    for path in tqdm(paths):
        cls_name = os.path.basename(path).split("Class")[1][0]  # クラス名の取得

        with open(path, "r", encoding="utf-8") as handle:
            for record in SeqIO.parse(handle, "fasta"):
                if check_valid_seq(record.seq):  # 有効な配列かどうか
                    writer.writerow([idx, cls_name])  # 行を追加

                    idx += 1

print(f"\033[32m✔ Saved CSV file to: \033[95m{csv_path}\033[0m")

100%|██████████| 87/87 [00:00<00:00, 1651.77it/s]

✔ Saved CSV file to: ../labels\bias-profs-labels.csv
